# 말이 달라도, 같은 동아리

`컴공`으로 개발 동아리를 찾는지 확인합니다. 검색 주소와 응답 형식은 그대로입니다.

**준비** · `sql/261109_club_semantic_search.sql` 적용 → [관리자 대시보드](http://localhost:3000/admin) → **검색 정보 갱신**.

동아리 소개는 관리자가 월 1회 반영하고, 검색어는 요청할 때 같은 모델로 변환합니다.

In [ ]:
BASE_URL = "http://localhost:3000"
QUERIES = ["컴공", "주식 공부", "함께 음악 연주"]

## 검색해보기

게스트 ID 헤더를 포함해 검색하고 상위 다섯 개만 봅니다. 검색어를 바꾸고 아래 셀을 다시 실행하세요.

In [ ]:
import json
from html import escape
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from uuid import uuid4
from IPython.display import HTML, display

headers = {"x-guest-id": str(uuid4())}
cards = []
for query in QUERIES:
    params = urlencode({"query": query, "record_recent_search": "false"})
    request = Request(f"{BASE_URL}/api/v2/clubs/search?{params}", headers=headers)
    with urlopen(request, timeout=30) as response:
        result = json.load(response)
    rows = "".join(
        f'<tr><td>{rank}</td><td><b>{escape(club["name"])}</b><br>'
        f'<span>{escape((club.get("shortDescription") or club.get("description") or "")[:140])}</span></td></tr>'
        for rank, club in enumerate(result["clubs"][:5], 1)
    )
    cards.append(f'<section><h3>{escape(query)} <small>{result["totalSize"]}개</small></h3>'
                 f'<table>{rows or "<tr><td>검색 결과가 없습니다.</td></tr>"}</table></section>')

display(HTML('''<style>
.club-search {font-family:system-ui,sans-serif;max-width:800px;color:#0f172a}
.club-search section {border:1px solid #e2e8f0;border-radius:12px;padding:20px;margin:16px 0;background:#fff}
.club-search h3 {margin:0 0 12px;font-size:20px}.club-search small {color:#64748b;font-size:12px;margin-left:8px}
.club-search table {width:100%;border-collapse:collapse}.club-search td {padding:12px 6px;border-top:1px solid #f1f5f9;text-align:left}
.club-search td:first-child {width:28px;color:#64748b}.club-search span {color:#64748b;font-size:13px}
</style><div class="club-search">''' + "".join(cards) + '</div>'))

## 결과를 읽는 법

- 문자열로 찾은 동아리를 먼저, 의미로 찾은 추가 동아리를 유사도순으로 표시합니다.
- 기존 모집·소속 필터와 공개 승인 조건은 의미 검색에도 적용됩니다.
- 새 소개가 검색에 반영되지 않았다면 대시보드에서 **갱신 필요** 개수를 확인하세요.

모델: [`text-embedding-3-small`](https://developers.openai.com/api/docs/models/text-embedding-3-small), 1,536차원. 벡터 생성 실패 시 기존 문자열 검색을 제공합니다.